# 10 — Gemini-assisted BM25 retrieval

Review scored iteration 9 after running `python evaluate_llm.py --config gemini`. This notebook never calls Gemini. It loads saved results and lets you run questions whose rewrites are already cached.

Gemini expands only the clause question; original full-query document routing and the best non-LLM BM25 configuration stay fixed. Original clause terms are retained alongside generated terms. Invalid model output falls back to the original query. Test data is not evaluated.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT=Path.cwd().resolve()
if ROOT.name=='notebooks':
    ROOT=ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
from improve import read_log, read_json
from llm_rewrite import RewriteRetriever, CacheMissError
from retrieval import BM25Retriever, load_corpus

runs=[r for r in read_log(ROOT/'experiments/llm_log.jsonl') if r['iteration']==9]
assert runs, 'Complete cache-only iteration 9 evaluation first.'
run=runs[-1]
print('Model:',run['model_config']['model'])
print('Cache output status:',run['cache_status_counts'])
print('Operational decision:',run['decision'])
print('Replay-only decision:',run['replay_decision'])
rows=[]
for label,result in run['metrics'].items():
    at5=next(m for m in result['retrieval'] if m['k']==5)
    rows.append(dict(approach=label,precision=at5['precision'],recall=at5['recall'],
                     benchmark_precision=at5['benchmark_precision'],benchmark_recall=at5['benchmark_recall'],
                     false_abstention=result['false_abstention'],correct_abstention=result['correct_abstention'],
                     replay_p95_ms=result['latency_ms']['p95']))
display(pd.DataFrame(rows))

## Cost and latency are separate from retrieval quality

Cache fill makes one generation request per unique clause question, reusing it across repeated question types. Token counts are provider usage, not an invoice. Actual charges depend on project billing and pricing.

Fast replay is not first-run inference speed. Estimated first-use latency combines the saved model-call time with measured cache/retrieval time for each question. The numerical latency guardrail is unchanged and applies to that first-use estimate in the operational decision. A retrieval gain can therefore still be rejected on latency. The estimate combines separately timed components; it is not a direct production load test.

In [ ]:
print('Unique cached requests:',run['unique_cached_requests'])
print('Provider-reported usage:',run['aggregate_unique_request_usage'])
print('Total cache-generation time (seconds):',run['generation_total_ms']/1000)
print('Estimated first-use latency (ms):',run['estimated_first_use_latency_ms'])
print('Billing:',run['billing_note'])
print('Original failures:',run['diagnosis_before'])
print('Candidate failures:',run['diagnosis_after'])

## Run the cached retrieval candidate

Edit the question only to a question type whose rewrite has been cached. Missing entries raise a clear error rather than secretly call a model. The contract description is used locally for routing and is not part of Gemini's clause-question input.

In [ ]:
documents,_=load_corpus(ROOT)
base=BM25Retriever(documents,run['reference_config'])
model=RewriteRetriever(base,ROOT,run['model_config'])
question='Consider the Wireless Content License Agreement between Fox Mobile Entertainment and Glu Mobile; What is the expiration date of this contract?'
try:
    hits=model.search(question,k=5)
    if hits:
        display(pd.DataFrame(hits)[['score','file_path','start','end','text']])
    else:
        print('No relevant passage found.')
except CacheMissError as error:
    print(error)

## Explain this step

“We tested whether a hosted model could add useful legal wording to a question before BM25 searches. We cached the exact model outputs so evaluation can be replayed without the API. We measured retrieval quality, failure handling, token usage, and both generation and replay latency separately.”

The result is limited to train data and this fixed prompt/model. It is not proof that an LLM answers legal questions correctly. Query expansion can cause drift; the original query and BM25 fallback remain available. Phi-3 will receive the same task/prompt only after the hosted checkpoint is complete. No FAISS or hybrid retrieval is used.